# Model Comparison: Exact versus approximate solutions

This notebook gives an overview of the differences between the `Mibitrans`, the `Anatrans` and the `Bioscreen` models, which reflect the use of a fully exact and fully analytical, but approximate solutions to the ADE for modelling 
contaminant transport. Guyonnet and Neville (2004)$^1$ studies the differences (originating from an integral approximation) and its causes. This notebook visualizse the amount and effects of the error. 

## Background

The `Mibitrans` model classes uses the exact solution of the 3D transport ADE, as described in Wexler et al. (1992) which requires numerical integration. The `Anatrans model` uses an approximate but fully analytical expression. Both models include the option for multiple source zones and source depletion.  

### Mibitrans model
The equation implemented in the `Mibitrans` model reads:
$$
\begin{equation}\tag{2}
\begin{aligned}
    C(x,y,z,t) &= \sum_{i=1}^{n}\left(C^*_{0,i}\frac{x}{8\sqrt{\pi \alpha_{x}v_R}}\exp(-\gamma_s t) \right. \\
    &\quad \cdot \int_{0}^{t}\left[\frac{1}{\tau^{\frac{3}{2}}} \exp\left((\gamma_s - \mu)\tau - \frac{(x-v_R\tau)^2}{4\alpha_{x}v_R\tau}\right) \right. \\
    &\quad \cdot \left\{\operatorname{erfc}\left(\frac{y-Y_i}{2 \sqrt{\alpha_{y}v_R\tau}}\right)-\operatorname{erfc}\left(\frac{y+Y_i}{2 \sqrt{\alpha_{y}v_R\tau}}\right) \right\} \\
    &\quad \left. \left. \cdot \left\{\operatorname{erfc}\left(\frac{-Z}{2 \sqrt{\alpha_{z}v_R\tau}}\right)-\operatorname{erfc}\left(\frac{Z}{2 \sqrt{\alpha_{z}v_R\tau}}\right) \right\}\right] d\tau \right)
\end{aligned}
\end{equation}
$$

### Anatrans model

The equation implemented in the `Anatrans` model reads:
\begin{align}\tag{1}
    C(x, y, z, t) &= \sum_{i=1}^{n}\left\{ \frac{C^*_{0,i}}{8} \exp \left(-\gamma_s t\right) \right. \\
    &\quad \cdot \left( \exp \left[ \frac{x\left(1-P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x - Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right] \right.\\
    &\quad \:  + \left.  \exp \left[ \frac{x\left(1+P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x + Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right] \right) \\
    &\quad \cdot \left( \operatorname{erf} \left[ \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right] - \operatorname{erf} \left[ \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right] \right) \\
    &\quad \cdot \left. \left( \operatorname{erf} \left[ \frac{Z}{2\sqrt{\alpha_z x)}} \right] - \operatorname{erf} \left[ \frac{-Z}{2\sqrt{\alpha_z x}} \right] \right) \right\} \\
    & \text{with} \quad P = \sqrt{1+4\left(\mu - \gamma_s \right) \alpha_x/v_R}
\end{align}

The `Anatrans` model equation can be derived from the exact `Mibitrans` model solution by splitting the effect of time on the different directions following the principle $C(x,y,z,t)/C_0 = C(x,t)\cdot C(y,x)\cdot C(z,x)/C_0$. Specifically, by applying the substitution $\tau = x/v_R$ (time traveled of advection front) in the error-function terms for the $y$ and $z$ directions. The remaining integral in $\tau$ in  the `Mibitrans` model for the transport in $x$-direction can be solved analytically resulting in the closed form expression in the `Anatrans` model.

### Bioscreen model

**ToDo**

### Approximation
The approximation for transverse dispersivity terms in the `Anatrans` model equation introduces an error. The size of the error depends on parameter choices. Guyonnet and Neville (2004)$^1$ studies the differences and its causes in more detail, using a dimensionless analysis. This notebook allows to visualize the amount and effects of the error introduced by the integral approximation. 

## Model Comparison Parameters

| **Quantity (Unit)** | **Values** |
|---|---|
| α_L (m) | 1; 3; **10** |
| α_T (m) | 0.002; **0.02**; 0.2 |
| α_V (m) | 0; 0.0005; **0.005**; 0.05 |
| Source zone boundaries, Y_i (m) | 1; **5**; 20 |


## References
$^1$ [Guyonnet, D., & Neville, C. (2004). *Dimensionless analysis of two analytical solutions for 3-D solute transport in groundwater*. Journal of Contaminant Hydrology, 75(1), 141–153.](https://doi.org/10.1016/j.jconhyd.2004.06.004)


## Model Setup and Parameter Definition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt
from mibitrans.visualize.animation import animate_1d

In [ ]:
###  Functions for quantitative comparison
import differences as df

In [ ]:
### Define options for plot modification 
colors_blue=['lightblue', 'cornflowerblue', 'navy']
# lcmap = plt.get_cmap('tab20c')
cmap = plt.get_cmap('tab20b')
colors_1 = cmap.colors   # list of RGB tuples

cmap = plt.get_cmap('Set1')
colors_2 = cmap.colors   # list of RGB tuples

ls_order = ['-','--',':','-.']
textsize = 8

### Define Imput parameters for standard values of dispersivity

Other standard settings:
* no decay
* standard example values

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.1,      # Flow velocity [m/d]
    porosity=0.3,      # Effective soil porosity [-]
    alpha_x=10,         # Longitudinal dispersivity, in [m]
    alpha_y=0.02,      # Transverse horizontal dispersivity, in [m]
    alpha_z=0.005,         # Transverse vertical dispersivity, in [m]
    diffusion=0,       # Molecular diffusion, in [m2/day]
)

att = mbt.AttenuationParameters(
    retardation=1,
    half_life=0,
)

source = mbt.SourceParameters(
    source_zone_boundary=np.array([5]),
    source_zone_concentration=np.array([10]),
    depth=2,
    total_mass="inf"
)

model = mbt.ModelParameters(
    model_length = 600,
    model_width = 40,
    model_time = 10*365,
    dx = 5,
    dy = 0.25,
    dt = 365//5,
)

In [ ]:
mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results = ana_object.run()

bio_model = mbt.Bioscreen(hydro, att, source, model)
bio_results = bio_model.run()

In [ ]:
# Plot the x and y concentration distribution for no degradation decay model, uses plot_surface
mbt_results.plume_3d()#time=5*365)
plt.show()

## Visual comparison for Standard Parameters

In [ ]:
times = mbt_results.t[4::20]
#print(times)
t1 = 365 #1y
t2 = 3*365 #3y
t3 = 9*365 # 9y
lw = 2.5

plt.figure(figsize = [3.75,2.75])
mbt_results.centerline(time=t1, color=colors_1[2],ls = '-',lw=lw, label=f"Mibitrans: $t_1$")
ana_results.centerline(time=t1, color=colors_1[2+8],ls = '--',lw=lw, label=f"Anatrans: $t_1$")
bio_results.centerline(time=t1, color=colors_1[2+12],ls = '-.',lw=lw, label=f"Bioscreen: $t_1$")

mbt_results.centerline(time=t2, color=colors_1[1],ls = '-',lw=lw, label=f"$t_2$")
ana_results.centerline(time=t2, color=colors_1[1+8],ls = '--',lw=lw, label=f"$t_2$")
bio_results.centerline(time=t2, color=colors_1[1+12],ls = '-.',lw=lw, label=f"$t_2$")

mbt_results.centerline(time=t3, color=colors_1[0],ls = '-',lw=lw, label=f"$t_3$")
ana_results.centerline(time=t3, color=colors_1[0+8],ls = '--',lw=lw, label=f"$t_3$")
bio_results.centerline(time=t3, color=colors_1[0+12],ls = '-.',lw=lw, label=f"$t_3$")

plt.legend(ncols = 3,fontsize = textsize,bbox_to_anchor=(0.2,0.85))
#plt.legend(ncols = 3,loc = 'upper right',fontsize = textsize)
plt.xlim((-1,420)) # Reduce the x-axis limit to better observe differences
plt.ylim((0,11))
plt.xlabel('Plume travel distance $x$',fontsize = textsize)
plt.ylabel('Concentration $C(x,y=0,t_i)$',fontsize = textsize)
plt.grid(True)
plt.title("")#Plume mass over time")
plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
plt.savefig('model_comparison_centerline.pdf')


In [ ]:
# plt.figure(figsize = [3.75,2.75])
xx = [20,110,350]

plt.figure(figsize = [3.75,2.75])
mbt_results.transverse(x_position = xx[0], time=t1, color=colors_1[2],ls = '-',lw=lw, label=f"Mibitrans, $t_1$")
ana_results.transverse(x_position = xx[0], time=t1, color=colors_1[2+8],ls = '--',lw=lw, label=f"Anatrans, $t_1$")
bio_results.transverse(x_position = xx[0], time=t1, color=colors_1[2+12],ls = '-.',lw=lw, label=f"Bioscreen, $t_1$")

mbt_results.transverse(x_position = xx[1], time=t2, color=colors_1[1],ls = '-',lw=lw, label=f"Mibitrans, $t_2$")
ana_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+8],ls = '--',lw=lw, label=f"Anatrans, $t_2$")
bio_results.transverse(x_position = xx[1], time=t2, color=colors_1[1+12],ls = '-.',lw=lw, label=f"Bioscreen, $t_2$")

mbt_results.transverse(x_position = xx[2], time=t3, color=colors_1[0],ls = '-',lw=lw, label=f"Mibitrans, $t_3$")
ana_results.transverse(x_position = xx[2], time=t3, color=colors_1[0+8],ls = '--',lw=lw, label=f"Anatrans, $t_3$")
bio_results.transverse(x_position = xx[2], time=t3, color=colors_1[0+12],ls = '-.',lw=lw, label=f"Bioscreen, $t_3$")

#plt.legend(bbox_to_anchor=(1,1),ncols = 3)
plt.ylim((0, 9))
plt.xlim((-10,10))
plt.grid(True)
plt.title("")#Plume mass over time")

plt.xlabel('Plume cross section $y$',fontsize = textsize)
plt.ylabel('Concentration $C(x_i,y,t_i)$',fontsize = textsize)

plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
plt.savefig('model_comparison_trans.pdf')


## Quantitative Comparison for Standard Parameters

In [ ]:
diff_mbt_bio = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:])
diff_mbt_ana = df.mean_absolute_difference(ana_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:])
diff_ana_bio = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], ana_results.relative_cxyt[:,:,:])

print("Mean absolute difference in relative concentration with standard example parameters (no decay):")
print("Between Mibitrans and Bioscreen model class",  diff_mbt_bio)
print("Between Mibitrans and Anatrans model class",  diff_mbt_ana)
print("Between Anatrans and Bioscreen model class",  diff_ana_bio)

In [ ]:
diff_mbt_bio_t = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:], mean_axis = (1,2))
diff_mbt_ana_t = df.mean_absolute_difference(ana_results.relative_cxyt[:,:,:], mbt_results.relative_cxyt[:,:,:], mean_axis = (1,2))
diff_ana_bio_t = df.mean_absolute_difference(bio_results.relative_cxyt[:,:,:], ana_results.relative_cxyt[:,:,:], mean_axis = (1,2))
#print(len(diff_mbt_bio_t), len(mbt_results.t))

In [ ]:
print("Maximum mean relative difference between Mibitrans and Bioscreen model:", np.max(diff_mbt_bio_t))
print("Maximum mean relative difference between Mibitrans and Anatrans model:", np.max(diff_mbt_ana_t))
print("Maximum mean relative difference between Anatrans and Bioscreen model:", np.max(diff_ana_bio_t))

In [ ]:
plt.figure(figsize = [3.75,2.5])
plt.plot(mbt_results.t/365,diff_mbt_bio_t*100,marker = '.',color = colors_2[4],ls = '-',label = f'Mibitrans vs Bioscreen')
plt.plot(mbt_results.t/365,diff_ana_bio_t*100,marker = '.',color = colors_2[3],ls = '-',label = f'Anatrans vs Bioscreen')
plt.plot(mbt_results.t/365,diff_mbt_ana_t*100,marker = '.',color = colors_2[2],ls = '-',label = f'Mibitrans vs Anatrans')

plt.legend(fontsize = textsize)
plt.xlabel('Travel time of Plume $t$ (years)',fontsize = textsize)
plt.ylabel('Mean relative difference in %',fontsize = textsize)
plt.xlim([0,10])
plt.grid(True)
plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
plt.savefig('model_comparison_diff_t.pdf')

In [ ]:
time_step = 14 #len(mbt_object.t)//2 - 1
print(time_step,mbt_object.t[time_step]/365)
print(mbt_object.disp_x,mbt_object.disp_y,mbt_object.disp_z)

diff_mbt_bio = bio_results.relative_cxyt[time_step,:,:] - mbt_results.relative_cxyt[time_step,:,:]
diff_mbt_ana = ana_results.relative_cxyt[time_step,:,:] - mbt_results.relative_cxyt[time_step,:,:]

In [ ]:
plt.figure(figsize = [3.75,2.5])

plt.pcolormesh(
    # mbt_object.x,
    # mbt_object.y,
    # diff_mbt_bio,
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_bio[:,:len(mbt_object.x)//4],
    # mbt_object.x[:len(mbt_object.x)//4],
    # mbt_object.y[len(mbt_object.y)//5:-len(mbt_object.y)//5],
    # diff_mbt_bio[len(mbt_object.y)//5:-len(mbt_object.y)//5,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)

cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference", fontsize=textsize)
cbar.ax.tick_params(labelsize=textsize)

plt.xlabel("Plume travel distance $x$ [m]", fontsize=textsize)
plt.ylabel('Plume cross section $y$ [m]',fontsize = textsize)
plt.ylim((-10,10))

# Change colorbar label size
plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
plt.savefig('model_comparison_mbt_bio_2D.pdf')

In [ ]:
plt.figure(figsize = [3.75,2.5])

plt.pcolormesh(
    # mbt_object.x,
    # mbt_object.y,
    # diff_mbt_bio,
    mbt_object.x[:len(mbt_object.x)//4],
    mbt_object.y,
    diff_mbt_ana[:,:len(mbt_object.x)//4],
    # mbt_object.x[:len(mbt_object.x)//4],
    # mbt_object.y[len(mbt_object.y)//5:-len(mbt_object.y)//5],
    # diff_mbt_bio[len(mbt_object.y)//5:-len(mbt_object.y)//5,:len(mbt_object.x)//4],
    vmin=-0.2,
    vmax=0.2,
    cmap="seismic",
    shading="gouraud"
)

cbar = plt.colorbar(orientation="vertical")
cbar.set_label("Absolute concentration difference", fontsize=textsize)
cbar.ax.tick_params(labelsize=textsize)

plt.xlabel("Plume travel distance $x$ [m]", fontsize=textsize)
plt.ylabel('Plume cross section $y$ [m]',fontsize = textsize)
plt.ylim((-10,10))

# Change colorbar label size
plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
plt.savefig('model_comparison_mbt_ana_2D.pdf')